# Construct the Path for the Analysis of the Inflow into the 79NG Cavity

This notebook constructs a path from the continental shelf into the 79NG cavity to analyse the dense inflow, based on the bathymetry and the simulated velocity.

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
import gsw
import numpy as np
import xarray as xr
from cmocean import cm
from matplotlib import pyplot as plt

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc").squeeze()

# Compute the flow speed
assert ds.velx3d.units == ds.vely3d.units, "units of u3d and v3d differ"
ds["vel3d"] = np.sqrt(ds.velx3d**2 + ds.vely3d**2)
ds.vel3d.attrs = {"long_name": "absolute value of the global velocity (3D)", "units": ds.velx3d.units}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

ds

## Part 1: path over the sill

### Determine the deepest points at the two bottlenecks

In the channel leading to the cavity, there are (at least) two bottlenecks.
At those bottlenecks, there is a unique deepest point that the densest flow must pass over, since a path over any other point would mean an inflow at shallower depth.
To see the two bottlenecks clearly, check out the figure further down in this notebook.

In [ ]:
lat0 = ds.latc.sel(latc=79.6, method="nearest")
lon0 = ds.bathymetry.sel(latc=lat0, lonc=slice(-19.5, -19)).idxmax()

lon1 = ds.lonc.sel(lonc=-19.5, method="nearest")
lat1 = ds.bathymetry.sel(lonc=lon1, latc=slice(79.5, 79.65)).idxmax()

fig, ax = plt.subplots()
ds.bathymetry.sel(lonc=slice(-19.8, -19), latc=slice(79.5, 79.65)).plot()
ax.set_title("Bathymetry around the sill")
for lat, lon in [(lat0, lon0), (lat1, lon1)]:
    print(f"{lat = :.4f}, {lon = :.4f}, depth = {ds.bathymetry.sel(latc=lat, lonc=lon):.2f} {ds.bathymetry.units}")
    ax.plot(lon, lat, "rx")
ax.set_aspect(grille_carree)
ax.grid(ls=":")

### Build a path with a smooth topography between these two points

The points of the path are chosen such that the resulting topography is relatively smooth, i.e., increases and decreases monotonically if possible.
This can be confirmed with the next figure.

In [ ]:
lon, lat = np.array([
    (lon0 + 5 * dlon, lat0 +10 * dlat),
    (lon0 + 5 * dlon, lat0 + 9 * dlat),
    (lon0 + 5 * dlon, lat0 + 8 * dlat),
    (lon0 + 4 * dlon, lat0 + 7 * dlat),
    (lon0 + 3 * dlon, lat0 + 6 * dlat),
    (lon0 + 2 * dlon, lat0 + 5 * dlat),
    (lon0 + 1 * dlon, lat0 + 4 * dlat),
    (lon0 + 1 * dlon, lat0 + 3 * dlat),
    (lon0 + 0 * dlon, lat0 + 2 * dlat),
    (lon0 + 0 * dlon, lat0 + 1 * dlat),
    (lon0, lat0),
    (lon0 + 0 * dlon, lat0 - 1 * dlat),
    (lon0 - 1 * dlon, lat0 - 2 * dlat),
    (lon0 - 1 * dlon, lat0 - 3 * dlat),
    (lon0 - 2 * dlon, lat0 - 4 * dlat),
    (lon0 - 3 * dlon, lat0 - 5 * dlat),
    (lon0 - 4 * dlon, lat0 - 6 * dlat),
    (lon0 - 5 * dlon, lat0 - 7 * dlat),
    (lon0 - 6 * dlon, lat1),
    (lon1, lat1),
    (lon1 - dlon, lat1),
])[::-1].T
path1 = xr.Dataset(
    {
        "lonc": (["i"], ds.lonc.sel(lonc=lon, method="nearest").data),
        "latc": (["i"], ds.latc.sel(latc=lat, method="nearest").data),
    },
    {"i": np.arange(lon.shape[0])},
)
path1

### Show this path and the bathymetry along it

In [ ]:
fig, axs = plt.subplots(2, height_ratios=[4, 1], constrained_layout=True, figsize=(8, 9))

ax = axs[0]
ds.bathymetry.where(ds.bathymetry > 300).sel(lonc=slice(-19.8, -19), latc=slice(79.52, 79.65)).plot(ax=ax, cmap=cm.deep)
cs = ds.bathymetry.plot.contour(ax=ax, levels=[325], colors="r", linewidths=1)
ax.plot([], [], color=cs.colors, lw=cs.get_linewidth(), label=f"{cs.levels[0]:.0f} {ds.bathymetry.units}-depth contour")
ax.plot(path1.lonc, path1.latc, "k.--", label="path over the sill")
for lat, lon, label, marker in [(lat0, lon0, "sill", "o"), (lat1, lon1, "bottleneck", "s")]:
    ax.scatter(lon, lat, edgecolor="k", color="none", linewidth=1, label=label, marker=marker)
ax.legend(loc="center right")
ax.set_title("Bathymetry around the sill (gray: shallower than 300 m)")
ax.set_facecolor("#ddd")
ax.set_aspect(grille_carree)

ax = axs[1]
(-ds.bathymetry).sel(path1).plot(ax=ax, marker="o")
ax.set_title("Elevation along the path")
ax.set_xlabel("Index along the path (west to east)")
ax.set_ylabel("Elevation [m]")
ax.set_xticks(np.arange(path1.i.size))
ax.grid()

## Part 2: path following inflow

The points of the path are chosen such that they follow the velocity maximum in the flow direction and do not go uphill.
This can be confirmed with the next figure.
For the velocity maximum, we consider the flow below 300 m depth, i.e., below the sill crest.

### Compute the maximum velocity below 300 m depth

In [ ]:
velx_deep = ds.velx3d.where(ds.zc < -300).sel(level=ds.vel3d.idxmax("level", fill_value=1))
vely_deep = ds.vely3d.where(ds.zc < -300).sel(level=ds.vel3d.idxmax("level", fill_value=1))
vel_deep_max = ds.vel3d.where(ds.zc < -300).max("level")
vel_deep_max.attrs = {"long_name": "max. velocity below 300 m", "units": ds.vel3d.units}

### Build a path

In [ ]:
lat2 = path1.latc.min()
lon2 = path1.lonc.min()

lon, lat = np.array([
    (lon2 - dlon, lat2),
    (lon2 - 2 * dlon, lat2),
    (lon2 - 3 * dlon, lat2),
    (lon2 - 4 * dlon, lat2 + 0 * dlat),
    (lon2 - 5 * dlon, lat2 + 1 * dlat),
    (lon2 - 6 * dlon, lat2 + 1 * dlat),
    (lon2 - 7 * dlon, lat2 + 1 * dlat),
    (lon2 - 8 * dlon, lat2 + 2 * dlat),
    (lon2 - 9 * dlon, lat2 + 3 * dlat),
])[::-1].T
path2 = xr.Dataset(
    {
        "lonc": (["i"], ds.lonc.sel(lonc=lon, method="nearest").data),
        "latc": (["i"], ds.latc.sel(latc=lat, method="nearest").data),
    },
    {"i": np.arange(lon.shape[0])},
)

### Show the path on a map of the maximum velocity

In [ ]:
fig, ax = plt.subplots(dpi=200)
ds.bathymetry.plot.contour(levels=np.arange(300, 510, 20), colors="grey", linewidths=0.5)
vel_deep_max.sel(latc=slice(79.52, 79.65), lonc=slice(-20.1, -19.1)).plot(cmap=cm.amp)
ax.quiver(ds.lonc, ds.latc, velx_deep, vely_deep, zorder=10, color="w")
ax.plot(path1.lonc, path1.latc, "k.--", lw=0.5, ms=6, label="path over the sill")
ax.plot(path2.lonc, path2.latc, "b.--", lw=0.5, ms=6, label="path following the inflow")
ax.legend(loc="lower left")
ax.set_title("Maximum velocity below 300 m and depth contours")
ax.set_aspect(grille_carree)

## Combine both parts and compute the distance along the path

In [ ]:
full_path = xr.Dataset(
    {
        "lonc": (
            ["i"],
            np.concatenate((path2.lonc, path1.lonc)),
            {"long_name": "longitude", "units": "°E"},
        ),
        "latc": (
            ["i"],
            np.concatenate((path2.latc, path1.latc)),
            {"long_name": "latitude", "units": "°N"},
        ),
    },
    {"i": np.arange(path2.i.size + path1.i.size)},
    {
        "title": "Path into the 79NG cavity",
        "author": "Markus Reinert (ORCID: 0000-0002-3761-8029)",
        "institution": "Leibniz Institute for Baltic Sea Research Warnemuende (IOW), Germany",
        "description": "Coordinates of a path over the sill following the inflow into the 79NG cavity on the GETM topography of the 79NG fjord.",
    },
)
# Replace index by distance coordinate
full_path = full_path.assign_coords(i=np.cumsum([0] + [
    gsw.distance([full_path.lonc[i], full_path.lonc[i+1]], [full_path.latc[i], full_path.latc[i+1]])[0]
    for i in range(full_path.i.size - 1)
]) / 1e3).rename(i="dist")
full_path.dist.attrs = {"long_name": "distance along the path", "units": "km"}
full_path

### Show the combined path and its bathymetry

In [ ]:
fig, axs = plt.subplots(2, height_ratios=[3, 1], constrained_layout=True, figsize=(6, 5), dpi=200)

ax = axs[0]
ds.bathymetry.plot.contour(ax=ax, levels=np.arange(0, 1000, 100), colors="grey", linewidths=0.5)
vel_deep_max.sel(latc=slice(79.5, 79.7), lonc=slice(-20.5, -19)).plot(ax=ax, cmap=cm.amp)
full_path.plot.scatter(ax=ax, x="lonc", y="latc", s=4, hue="dist", lw=0.1, edgecolor="k", cmap=cm.ice_r)
ax.set_title("Selected path")
ax.set_aspect(grille_carree)

ax = axs[1]
(-ds.bathymetry).sel(full_path).plot(ax=ax, marker=".")
ax.set_title("Elevation along the path")
ax.set_ylabel("Elevation [m]")
ax.set_yticks(np.arange(-450, -250, 50))
ax.set_xlim(0, full_path.dist.max())

### Save the path

In [ ]:
full_path.to_netcdf("data/path_inflow.nc")